In [9]:
import torch, torch.nn as nn, torch.nn.functional as F
from transformers import GPT2Tokenizer, GPT2Config

In [2]:
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

In [6]:
# Testing GPT2Tokenizer model
tokens = tokenizer.tokenize("what is your name")
tokens = [tokenizer.bos_token] + tokens + [tokenizer.eos_token]
tokens

['<|endoftext|>', 'what', 'Ġis', 'Ġyour', 'Ġname', '<|endoftext|>']

In [10]:
class GPT2Attention(nn.Module):
    def __init__(self, config):
        super(GPT2Attention, self).__init__()
        max_positions = config.n_positions
        self.mask = torch.tril(torch.ones((1, max_positions, max_positions)), dtype = torch.uint8).unsqueeze(0).unsqueeze(0)
        self.embed_dim = config.n_embed
        self.num_heads = config.n_head
        self.head_dim = self.embed_dim // self.num_heads
        self.split_size = self.embed_dim
        self.c_attn = nn.Linear(self.embed_dim, 3*self.embed_dim)
        self.c_proj = nn.Linear(self.embed_dim, self.embed_dim)
        self.dropout = nn.Dropout(0.1)

    def _attn(self, query, key, value):
        # query, key, value: [B, nh, seq_len, d_head]

        # [B, nh, seq_len, d_head] matmul [B, nh, d_head, seq_len] -> [B, nh, seq_len, seq_len]
        attn_weights = torch.matmul(query, key.transpose(-1, -2))
        attn_weights /= float(value.size(-1)) ** 0.5

        T = query.size(-2)
        casual_mask = self.mask[:,:,:T,:T].bool()
        attn_weights = torch.where(casual_mask, attn_weights, float(-1e4))
        attn_weights = F.softmax(attn_weights, dim = -1)
        attn_weights = self.dropout(attn_weights)

        # [B, nh, seq_len, seq_len] matmul [B, nh, seq_len, d_head] -> [B, nh, seq_len, d_head]
        attn_output = torch.matmul(attn_weights, value)
        return attn_output

    def forward(self, x):
        # x: [B,T,C]
        # T (time steps) is same as seq_len, C == self.embed_dim

        B,T,C = x.size()
        query, key, value = self.c_attn(x).split(self.split_size, dim = -1)

        # [B,nh,T,d_head]
        query = query.view(B, T, self.num_heads, self.head_dim).transpose(1,2)
        key = key.view(B, T, self.num_heads, self.head_dim).transpose(1,2)
        value = value.view(B, T, self.num_heads, self.head_dim).transpose(1,2)

        attn_output = self._attn(query, key, value) # [B, nh, seq_len, d_head]
        attn_output = attn_output.transpose(1,2).contiguous().view(B, T, C) # [B,T,C]
        attn_output = self.c_proj(attn_output) # [B,T,C]
        attn_output = self.dropout(attn_output)
        return attn_output

In [11]:
class GPT2MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        embed_dim = config.n_embd
        self.mlp = nn.Sequential(nn.Linear(embed_dim, 4*embed_dim),
                                 nn.GELU(),
                                 nn.Linear(4*embed_dim, embed_dim),
                                 nn.Dropout(0.1))

    def forward(self, x):
        return self.mlp(x)

In [12]:
class GPT2Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        embed_dim = config.n_embd
        self.ln_1 = nn.LayerNorm(embed_dim)
        self.ln_2 = nn.LayerNorm(embed_dim)
        self.attn = GPT2Attention(config)
        self.mlp = GPT2MLP(config)

    def forward(self, hidden_state):
        residual = hidden_state.clone()
        hidden_states = self.ln_1(hidden_state)
        attn_outputs = self.attn(hidden_states)
        hidden_states = residual + attn_outputs

        residual = hidden_states
        feed_forward_hidden_states = self.mlp(hidden_states)
        hidden_states = residual + feed_forward_hidden_states
        return hidden_states # [B, T, C]

In [14]:
class GPT2Model(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.embed_dim = config.n_embd
        self.vocab_size = config.vocab_size

        self.wte = nn.Embedding(self.vocab_size, self.embed_dim)
        self.wpe = nn.Embedding(config.n_positions, self.embed_dim)

        self.dropout = nn.Dropout(0.1)
        self.blocks = nn.ModuleList([GPT2Block(config) for _ in range(config.n_layer)])
        self.ln_f = nn.LayerNorm(self.embed_dim)

        self.init_weights()  # 🔥 gọi khởi tạo

    def init_weights(self):
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Embedding):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)
            elif isinstance(module, nn.LayerNorm):
                nn.init.ones_(module.weight)
                nn.init.zeros_(module.bias)

    def forward(self, input_ids = None, position_ids = None):
        # input_ids: [batch_size, seq_len]
        input_shape = input_ids.size()
        batch_size = input_ids.size(0)
        device = input_ids.device

        if position_ids is None:
            position_ids = torch.arrange(0, input_ids.size(-1), dtype = torch.long, device = device)
        position_ids = position_ids.unsqueeze(0)

        input_embeds = self.wte(input_ids)
        position_embeds = self.wpe(position_ids)
        hidden_states = input_embeds + position_embeds
        hidden_states = self.dropout(hidden_states)

        for block in self.blocks:
            hidden_states = block(hidden_states)

        hidden_states = self.ln_f(hidden_states)
        return hidden_states

In [15]:
class GPT2LMHead(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.transformer = GPT2Model(config)
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias = False)
        self.xe = nn.CrossEntropyLoss(ignore_index = tokenizer.pad_token)

    def forward(self, input_ids = None, position_ids = None, labels = None):
        hidden_states = self.transformer(input_ids, position_ids)
        lm_logits = self.lm_head(hidden_states) # [B, seq_len, vocab_size]

        loss = None
        if labels is not None:
            shift_logits = lm_logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()
            loss = self.xe(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))

        return lm_logits, loss

    @torch.no_grad()
    def generate(self, input_ids, max_length=50, temperature=1.0):
        self.eval()
        device = next(self.parameters()).device
        input_ids = input_ids.to(device)

        for _ in range(max_length):
            outputs = self.transformer(input_ids)
            logits = self.lm_head(outputs)  # [B, seq_len, vocab_size]
            next_token_logits = logits[:, -1, :] / temperature

            probs = F.softmax(next_token_logits, dim=-1)

            next_token = torch.argmax(probs, dim=-1).unsqueeze(-1)

            input_ids = torch.cat([input_ids, next_token], dim=-1)

        return input_ids